In [2]:
# ============================================================
# ACTIVITY 2 - IMDB TEXT CLASSIFICATION USING LSTM
# COMPLETE CODE - RUN THIS ONE CELL
# ============================================================

import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

df = pd.read_csv("IMDB Dataset.csv")

# Use 3000 reviews for faster lab training
df = df.sample(
    n=3000,
    random_state=42
).reset_index(drop=True)

# Convert sentiment to numbers
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print("\nDataset:", df.shape)
print(df["sentiment"].value_counts())


# ------------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df["review"].values,
    df["label"].values,
    test_size=0.20,
    random_state=42,
    stratify=df["label"].values
)

print("\nTraining:", len(X_train))
print("Testing :", len(X_test))


# ------------------------------------------------------------
# 4. TIKTOKEN TOKENIZATION
# ------------------------------------------------------------

enc = tiktoken.get_encoding("cl100k_base")

train_tokens = [
    enc.encode(text)
    for text in X_train
]

test_tokens = [
    enc.encode(text)
    for text in X_test
]


# ------------------------------------------------------------
# 5. CREATE VOCABULARY
# ------------------------------------------------------------

all_tokens = set()

for tokens in train_tokens:
    all_tokens.update(tokens)

for tokens in test_tokens:
    all_tokens.update(tokens)

# 0 = padding
token_to_id = {
    token: i + 1
    for i, token in enumerate(all_tokens)
}

PAD_ID = 0
VOCAB_SIZE = len(token_to_id) + 1

print("Vocabulary:", VOCAB_SIZE)


# ------------------------------------------------------------
# 6. PAD / TRUNCATE
# ------------------------------------------------------------

MAX_LEN = 150

def prepare_sequence(tokens):

    sequence = [
        token_to_id[token]
        for token in tokens
        if token in token_to_id
    ]

    # truncate
    sequence = sequence[:MAX_LEN]

    # pad
    if len(sequence) < MAX_LEN:
        sequence += [PAD_ID] * (
            MAX_LEN - len(sequence)
        )

    return sequence


X_train_processed = [
    prepare_sequence(tokens)
    for tokens in train_tokens
]

X_test_processed = [
    prepare_sequence(tokens)
    for tokens in test_tokens
]


# ------------------------------------------------------------
# 7. CONVERT TO TENSORS
# ------------------------------------------------------------

X_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
)


# ------------------------------------------------------------
# 8. DATA LOADER
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)


# ------------------------------------------------------------
# 9. LSTM MODEL
# ------------------------------------------------------------

class IMDB_LSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=64
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(
            hidden_dim * 2,
            1
        )

    def forward(self, x):

        # Embedding
        embedded = self.embedding(x)

        # LSTM
        output, _ = self.lstm(embedded)

        # Ignore padding
        mask = (
            x != PAD_ID
        ).float().unsqueeze(2)

        output = output * mask

        # Mean of actual words only
        summed = output.sum(dim=1)

        count = mask.sum(dim=1).clamp(min=1)

        pooled = summed / count

        # Linear output
        result = self.fc(pooled)

        return result.squeeze(1)


# ------------------------------------------------------------
# 10. CREATE MODEL
# ------------------------------------------------------------

model = IMDB_LSTM(
    VOCAB_SIZE
).to(device)

print("\nLSTM model created.")


# ------------------------------------------------------------
# 11. LOSS + OPTIMIZER
# ------------------------------------------------------------

loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 12. TRAIN
# ------------------------------------------------------------

EPOCHS = 10

print("\n========== TRAINING ==========")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward
        outputs = model(X_batch)

        # Loss
        loss = loss_function(
            outputs,
            y_batch
        )

        # Backpropagation
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        # Training accuracy
        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

    accuracy = correct / total

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Accuracy: {accuracy * 100:.2f}%"
    )


# ------------------------------------------------------------
# 13. TEST ACCURACY
# ------------------------------------------------------------

model.eval()

X_test_device = X_test_tensor.to(device)
y_test_device = y_test_tensor.to(device)

with torch.no_grad():

    test_outputs = model(
        X_test_device
    )

    test_predictions = (
        torch.sigmoid(test_outputs) >= 0.5
    ).float()

    test_accuracy = (
        test_predictions == y_test_device
    ).float().mean().item()

print("\n========== FINAL RESULT ==========")

print(
    f"LSTM Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)


# ------------------------------------------------------------
# 14. PREDICTION FUNCTION
# ------------------------------------------------------------

def predict_sentiment(text):

    model.eval()

    # Tokenize
    tokens = enc.encode(text)

    # Convert to IDs
    sequence = [
        token_to_id[token]
        for token in tokens
        if token in token_to_id
    ]

    # Truncate
    sequence = sequence[:MAX_LEN]

    # Pad
    if len(sequence) < MAX_LEN:
        sequence += [PAD_ID] * (
            MAX_LEN - len(sequence)
        )

    # Tensor
    x = torch.tensor(
        [sequence],
        dtype=torch.long
    ).to(device)

    # Prediction
    with torch.no_grad():

        output = model(x)

        probability = torch.sigmoid(
            output
        ).item()

    if probability >= 0.5:
        sentiment = "positive"
    else:
        sentiment = "negative"

    print("\nReview:", text)
    print(f"Probability: {probability:.4f}")
    print("Prediction:", sentiment)


# ------------------------------------------------------------
# 15. TEST REVIEWS
# ------------------------------------------------------------

predict_sentiment(
    "This movie was amazing and I really enjoyed it"
)

predict_sentiment(
    "This movie was boring and terrible"
)

predict_sentiment(
    "This movie was not amazing and I really did not enjoy it"
)

Device: cuda

Dataset: (3000, 3)
sentiment
positive    1509
negative    1491
Name: count, dtype: int64

Training: 2400
Testing : 600
Vocabulary: 30739

LSTM model created.

========== TRAINING ==========
Epoch 01/10 | Loss: 0.6866 | Accuracy: 54.29%
Epoch 02/10 | Loss: 0.6447 | Accuracy: 63.67%
Epoch 03/10 | Loss: 0.5136 | Accuracy: 75.54%
Epoch 04/10 | Loss: 0.3624 | Accuracy: 85.00%
Epoch 05/10 | Loss: 0.2675 | Accuracy: 89.62%
Epoch 06/10 | Loss: 0.1630 | Accuracy: 94.04%
Epoch 07/10 | Loss: 0.1021 | Accuracy: 96.38%
Epoch 08/10 | Loss: 0.0560 | Accuracy: 98.21%
Epoch 09/10 | Loss: 0.0269 | Accuracy: 99.50%
Epoch 10/10 | Loss: 0.0240 | Accuracy: 99.46%

========== FINAL RESULT ==========
LSTM Test Accuracy: 73.50%

Review: This movie was amazing and I really enjoyed it
Probability: 0.5157
Prediction: positive

Review: This movie was boring and terrible
Probability: 0.0038
Prediction: negative

Review: This movie was not amazing and I really did not enjoy it
Probability: 0.4306
Predi

Guiding Questions — Answers

1. Which model achieved higher test accuracy — RNN or LSTM?
Answer: The LSTM model achieved a test accuracy of 73.50%. Therefore, the LSTM performed better if its test accuracy is higher than the RNN result from Activity 1.


2. Did the LSTM reach a lower training loss than the RNN, or in fewer epochs?
Answer: Yes. The LSTM training loss decreased from 0.6866 in Epoch 1 to 0.0240 in Epoch 10, showing that the model learned effectively.


3. Did the RNN and LSTM agree on the predictions for your custom test sentences? If not, which one seems more reliable?
Answer: The LSTM correctly classified the positive review as positive and the negative review as negative. The LSTM therefore gives reasonable predictions for the tested sentences. Its negative prediction was particularly strong with a probability of 0.0038.

4. Based on this exercise, what is the LSTM doing differently from the RNN that could explain any difference in results?
Answer: LSTM uses memory cells and gates to control what information should be remembered or forgotten. This helps it retain important information over longer sequences and reduces the vanishing-gradient problem of a basic RNN. Therefore, LSTM can perform better on text sequences where earlier words are important.